# Phase 04B — Traditional Regression Baselines

Initialization and frozen-input preflight only. No model is instantiated, fitted, tuned, or evaluated in this notebook.

In [1]:
from pathlib import Path
import hashlib, json, platform, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import scipy, sklearn, joblib, threadpoolctl
from sklearn.model_selection import GroupKFold

PROJECT_ROOT = Path(r'E:\hdc-vr-pilot')
PHASE_DIR = PROJECT_ROOT / 'experiments' / 'phase_04b_traditional_regression_baselines'
PRIMARY_PATH = PROJECT_ROOT / 'experiments' / 'phase_03_multimodal_dataset_labeling' / 'data' / 'primary_without_performance.csv'
FOLD_PATH = PROJECT_ROOT / 'experiments' / 'phase_03_multimodal_dataset_labeling' / 'data' / 'fold_assignments.csv'
EXPECTED_SHA256 = 'e4dc943af21851bace49345f6336f9c88f82613ca3a26b3c433efe7dfb041f6f'
NON_FEATURE_COLUMNS = ['subject_id', 'session_id', 'run_id', 'difficulty_level_raw', 'difficulty_level', 'run_key', 'target_class', 'target_score', 'outer_fold']
SHARED_COLUMNS = NON_FEATURE_COLUMNS
assert PROJECT_ROOT.is_dir() and PRIMARY_PATH.is_file() and FOLD_PATH.is_file()
actual_sha256 = hashlib.sha256(FOLD_PATH.read_bytes()).hexdigest()
assert actual_sha256 == EXPECTED_SHA256, f'Frozen fold checksum mismatch: {actual_sha256}'
print('FROZEN FOLD CHECKSUM: PASS')
print('Phase 03 inputs loaded read-only from absolute paths.')

FROZEN FOLD CHECKSUM: PASS
Phase 03 inputs loaded read-only from absolute paths.


In [2]:
primary = pd.read_csv(PRIMARY_PATH)
folds = pd.read_csv(FOLD_PATH)
failures = []
def check(condition, label):
    if not bool(condition):
        failures.append(label)
    return bool(condition)

rows = len(primary); subjects = primary['subject_id'].nunique(); total_columns = len(primary.columns)
non_feature_present = [c for c in NON_FEATURE_COLUMNS if c in primary.columns]
predictive_features = total_columns - len(NON_FEATURE_COLUMNS)
target_values = sorted(primary['target_score'].dropna().unique().tolist())
target_missing = int(primary['target_score'].isna().sum())
unique_run_keys = int(primary['run_key'].nunique(dropna=True))
check(rows == 419, 'modeling rows != 419'); check(subjects == 35, 'subjects != 35'); check(total_columns == 1185, 'total columns != 1185')
check(len(NON_FEATURE_COLUMNS) == 9 and non_feature_present == NON_FEATURE_COLUMNS, 'specified non-feature columns invalid')
check(predictive_features == 1176, 'primary predictive features != 1176')
check(target_values == [1.0, 2.0, 3.0, 4.0], 'target_score values invalid'); check(target_missing == 0, 'target_score has missing values')
check(primary['run_key'].notna().all() and unique_run_keys == 419, 'primary run_key invalid')
check(len(folds) == 419, 'fold assignment rows != 419')
fold_unique_run_keys = int(folds['run_key'].nunique(dropna=True))
check(folds['run_key'].notna().all() and fold_unique_run_keys == 419, 'fold run_key invalid')
primary_duplicates = int(primary['run_key'].duplicated().sum()); fold_duplicates = int(folds['run_key'].duplicated().sum())
primary_keys, fold_keys = set(primary['run_key']), set(folds['run_key'])
missing_from_folds = len(primary_keys - fold_keys); extra_in_folds = len(fold_keys - primary_keys)
alignment_pass = check(missing_from_folds == 0 and extra_in_folds == 0 and primary_duplicates == 0 and fold_duplicates == 0, 'run_key alignment failure')
joined = primary[SHARED_COLUMNS].merge(folds[SHARED_COLUMNS], on='run_key', how='inner', validate='one_to_one', suffixes=('_primary', '_fold'))
shared_mismatches = {}
for column in SHARED_COLUMNS:
    if column != 'run_key':
        count = int((joined[f'{column}_primary'] != joined[f'{column}_fold']).sum())
        shared_mismatches[column] = count
shared_pass = check(len(joined) == 419 and all(v == 0 for v in shared_mismatches.values()), 'shared fields inconsistent')
outer_folds = int(folds['outer_fold'].nunique(dropna=True))
check(folds['outer_fold'].notna().all() and outer_folds == 5, 'outer folds invalid')
print(f'Rows={rows}; subjects={subjects}; columns={total_columns}; primary features={predictive_features}')
print(f'Run-key alignment={"PASS" if alignment_pass else "FAIL"}; shared-field consistency={"PASS" if shared_pass else "FAIL"}')

Rows=419; subjects=35; columns=1185; primary features=1176
Run-key alignment=PASS; shared-field consistency=PASS


In [3]:
outer_isolation = {}; inner_feasibility = {}; inner_pass = True
for fold_value in sorted(folds['outer_fold'].unique()):
    test_rows = folds[folds['outer_fold'] == fold_value]
    train_rows = folds[folds['outer_fold'] != fold_value]
    overlap = sorted(set(train_rows['subject_id']) & set(test_rows['subject_id']))
    outer_isolation[str(fold_value)] = {'train_subjects': int(train_rows['subject_id'].nunique()), 'test_subjects': int(test_rows['subject_id'].nunique()), 'subject_overlap': overlap, 'pass': len(overlap) == 0}
    unique_train_subjects = int(train_rows['subject_id'].nunique())
    splits = []; fold_inner_pass = unique_train_subjects >= 3
    if fold_inner_pass:
        for inner_index, (train_idx, val_idx) in enumerate(GroupKFold(n_splits=3).split(train_rows, groups=train_rows['subject_id']), start=1):
            train_subjects = set(train_rows.iloc[train_idx]['subject_id']); val_subjects = set(train_rows.iloc[val_idx]['subject_id'])
            split_overlap = sorted(train_subjects & val_subjects)
            split_pass = len(split_overlap) == 0
            fold_inner_pass = fold_inner_pass and split_pass
            splits.append({'inner_fold': inner_index, 'train_subjects': len(train_subjects), 'validation_subjects': len(val_subjects), 'subject_overlap': split_overlap, 'pass': split_pass})
    inner_feasibility[str(fold_value)] = {'outer_training_unique_subjects': unique_train_subjects, 'generated_inner_splits': len(splits), 'splits': splits, 'pass': fold_inner_pass}
    inner_pass = inner_pass and fold_inner_pass
outer_pass = check(all(x['pass'] for x in outer_isolation.values()), 'outer subject isolation failure')
inner_pass = check(inner_pass, 'inner GroupKFold feasibility failure')
print(f'Outer subject isolation: {"PASS" if outer_pass else "FAIL"}')
print(f'Inner 3-fold GroupKFold feasibility: {"PASS" if inner_pass else "FAIL"}')

Outer subject isolation: PASS
Inner 3-fold GroupKFold feasibility: PASS


In [4]:
frozen_environment = {'Python': r'D:\Computer\anaconda3\python.exe', 'PYTHONNOUSERSITE': '1', 'NumPy': '1.26.4', 'pandas': '2.2.2', 'SciPy': '1.13.1', 'scikit-learn': '1.5.1', 'joblib': '1.4.2', 'threadpoolctl': '3.5.0'}
actual_environment = {'Python': sys.executable, 'PYTHONNOUSERSITE': __import__('os').environ.get('PYTHONNOUSERSITE'), 'NumPy': np.__version__, 'pandas': pd.__version__, 'SciPy': scipy.__version__, 'scikit-learn': sklearn.__version__, 'joblib': joblib.__version__, 'threadpoolctl': threadpoolctl.__version__, 'platform': platform.platform()}
environment_differences = {key: {'frozen': value, 'actual': actual_environment.get(key)} for key, value in frozen_environment.items() if str(value) != str(actual_environment.get(key))}
contract = {'phase': 'Phase 04B Traditional Regression Baselines', 'phase03_frozen_fold_sha256': EXPECTED_SHA256, 'primary_data_path': str(PRIMARY_PATH), 'fold_assignments_path': str(FOLD_PATH), 'expected_rows': 419, 'expected_subjects': 35, 'predictive_features': 1176, 'outer_cv': 'frozen 5-fold subject-wise outer CV', 'inner_cv': 'GroupKFold(n_splits=3, groups=subject_id)', 'task': 'regression', 'regression_target': 'target_score = difficulty_level', 'target_interpretation': 'bounded difficulty-induced workload proxy regression', 'primary_metric': 'MAE', 'data_leakage_prohibitions': ['Do not regenerate or modify outer folds.', 'Fit imputation, missing indicators, variance filtering, scaling, feature selection, and tuning only in corresponding training folds.', 'Outer-test folds are final-test only.', 'Use Primary without-performance data for primary results.', 'Preserve prediction_raw and clip primary prediction_bounded to [1, 4] without rounding before evaluation.'], 'current_status': 'INITIALIZED / NOT YET MODELED'}
search_space = {'status': 'NOT_STARTED', 'hyperparameter_grids': 'PENDING_FREEZE', 'models': [{'name': name, 'status': 'NOT_STARTED'} for name in ['Dummy Regressor mean', 'Dummy Regressor median', 'Ridge', 'Elastic Net', 'Linear SVR', 'RBF SVR', 'Random Forest Regressor', 'Gradient Boosting Regressor']]}
audit = {'input_files': {'primary_data_path': str(PRIMARY_PATH), 'fold_assignments_path': str(FOLD_PATH)}, 'fold_sha256': actual_sha256, 'actual_data_rows': rows, 'subject_count': subjects, 'total_columns': total_columns, 'non_feature_columns': NON_FEATURE_COLUMNS, 'predictive_feature_count': predictive_features, 'target_score_values': target_values, 'target_score_missing': target_missing, 'unique_run_key_count': unique_run_keys, 'fold_assignment_rows': len(folds), 'fold_assignment_unique_run_key_count': fold_unique_run_keys, 'run_key_alignment': {'missing_from_fold_assignments': missing_from_folds, 'extra_in_fold_assignments': extra_in_folds, 'primary_duplicate_run_keys': primary_duplicates, 'fold_assignment_duplicate_run_keys': fold_duplicates, 'pass': alignment_pass}, 'shared_field_consistency': {'fields': SHARED_COLUMNS, 'mismatch_counts': shared_mismatches, 'pass': shared_pass}, 'outer_fold_count': outer_folds, 'outer_subject_isolation': outer_isolation, 'inner_groupkfold_feasibility': inner_feasibility, 'notebook_persistence': {'status': 'PENDING_EXTERNAL_REOPEN_CHECK', 'pass': None}, 'overall_pass_before_notebook_persistence': len(failures) == 0, 'overall_pass': False, 'failures': failures, 'utc_timestamp': datetime.now(timezone.utc).isoformat()}
def save_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + '\n', encoding='utf-8')
save_json(PHASE_DIR / 'configs' / 'phase04b_experiment_contract.json', contract)
save_json(PHASE_DIR / 'configs' / 'phase04b_environment.json', {'frozen_environment': frozen_environment, 'actual_environment': actual_environment, 'differences': environment_differences})
save_json(PHASE_DIR / 'configs' / 'regression_model_search_space.json', search_space)
save_json(PHASE_DIR / 'audits' / 'phase04b_input_and_fold_audit.json', audit)
print('Precheck artifacts saved; no regression models were instantiated or trained.')
print('Phase Validation Summary')
print('VERIFIED:', 'all input and fold checks passed' if not failures else 'FAILURES: ' + '; '.join(failures))
print('NOT VERIFIED: modeling results (intentionally not started)')
print('WARNINGS: environment differences recorded' if environment_differences else 'WARNINGS: none')
print('KEY RESULTS: initialization/precheck only')
print('OUTPUT FILES: audit and three configuration JSON files')
print('NEXT PHASE REQUIREMENTS: successful persisted-notebook reopen check')
print('FINAL STATUS CELL: PHASE 04B READY FOR MODELING: PENDING NOTEBOOK PERSISTENCE CHECK')

Precheck artifacts saved; no regression models were instantiated or trained.
Phase Validation Summary
VERIFIED: all input and fold checks passed
NOT VERIFIED: modeling results (intentionally not started)
WARNINGS: none
KEY RESULTS: initialization/precheck only
OUTPUT FILES: audit and three configuration JSON files
NEXT PHASE REQUIREMENTS: successful persisted-notebook reopen check
FINAL STATUS CELL: PHASE 04B READY FOR MODELING: PENDING NOTEBOOK PERSISTENCE CHECK
